In [90]:
!pip install pyspark

In [91]:
from pyspark.sql import SparkSession

spark=SparkSession.builder.appName("Working with files").getOrCreate()

In [ ]:
from google.colab import files

uploaded=files.upload()

Saving customer_ex.json to customer_ex (1).json


In [92]:
df = spark.read.option("multiline","true").json("customer_ex.json")

In [93]:
df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [94]:
df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [124]:
from pyspark.sql.functions import col,count,when

flat_df = df.select(
"customer_id",
"name",
"city",
"membership",
col("contact.phone").alias("phone"),
col("contact.email").alias("email"),
col("preferences.preferred_service").alias("preferred_service"),
col("preferences.budget_range").alias("budget_range")
)
flat_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [125]:
flat_df.select("name","city","phone","email").show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [126]:
flat_df.filter(flat_df.city.isNull()).show()

+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|customer_id|       name|city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+
|          3|John Mathew|NULL|      Gold|9876500013| NULL|           Flight|        High|
+-----------+-----------+----+----------+----------+-----+-----------------+------------+



In [127]:
flat_df.filter(flat_df.phone.isNull()).show()

+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|      name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+
|          2| Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
|          5|Vikram Rao|   Mumbai|  Platinum| NULL|         NULL|           Flight|        High|
+-----------+----------+---------+----------+-----+-------------+-----------------+------------+



In [128]:
flat_df.filter(flat_df.email.isNull()).show()

+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|customer_id|       name|  city|membership|     phone|email|preferred_service|budget_range|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+
|          3|John Mathew|  NULL|      Gold|9876500013| NULL|           Flight|        High|
|          5| Vikram Rao|Mumbai|  Platinum|      NULL| NULL|           Flight|        High|
+-----------+-----------+------+----------+----------+-----+-----------------+------------+



In [129]:
flat_df.filter(flat_df.membership.isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [130]:
flat_df.filter(flat_df.preferred_service.isNull()).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [131]:
flat_df.filter(flat_df.budget_range.isNull()).show()

+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|customer_id|     name|     city|membership|phone|        email|preferred_service|budget_range|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+
|          2|Sana Khan|Bangalore|    Silver| NULL|sana@mail.com|            Hotel|        NULL|
+-----------+---------+---------+----------+-----+-------------+-----------------+------------+



In [132]:
flat_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in flat_df.columns
]).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [133]:
flat_df = flat_df.na.fill({
    "city":"Unknown",
    "membership":"Standard",
    "phone":"Not Provided",
    "email":"Not Provided",
    "preferred_service":"Not Selected",
    "budget_range":"Unknown"
})

flat_df.show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [134]:
flat_df = flat_df.withColumn(
    "customer_quality_status",
    when(
        (col("city")=="Unknown") |
        (col("phone")=="Not Provided") |
        (col("email")=="Not Provided") |
        (col("membership")=="Standard") |
        (col("preferred_service")=="Not Selected"),
        "Incomplete"
    ).otherwise("Complete")
)

flat_df.show()


+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|             Incomplete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|

In [135]:
flat_df.groupBy("customer_quality_status").count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+



In [136]:
flat_df.filter(flat_df.customer_quality_status=="Complete").show()

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|       name|     city|membership|     phone|         email|preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|          1|Aarav Mehta|Hyderabad|      Gold|9876500011|aarav@mail.com|           Flight|      Medium|               Complete|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+



In [137]:
flat_df.filter(flat_df.customer_quality_status=="Incomplete").show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+-----------------------+
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|     Unknown|             Incomplete|
|          3| John Mathew|  Unknown|      Gold|  9876500013|   Not Provided|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|  Standard|  9876500014|ayesha@mail.com|     Not Selected|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|   Not Provided|           Flight|        High|             Incomplete|
+-----------+------------+---------+----------+------------+---------------+

In [138]:
flat_df.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [139]:
flat_df.groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [140]:
flat_df.write.mode("overwrite") \
    .parquet("customers_flat.parquet")

In [141]:
print("Original:", df.count())
print("Clean:", flat_df.count())

Original: 5
Clean: 5


In [144]:
flat_df.filter(
   (flat_df.phone=="Not Provided") |
    (flat_df.email=="Not Provided")
).show()

+-----------+-----------+---------+----------+------------+-------------+-----------------+------------+-----------------------+
|customer_id|       name|     city|membership|       phone|        email|preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+------------+-------------+-----------------+------------+-----------------------+
|          2|  Sana Khan|Bangalore|    Silver|Not Provided|sana@mail.com|            Hotel|     Unknown|             Incomplete|
|          3|John Mathew|  Unknown|      Gold|  9876500013| Not Provided|           Flight|        High|             Incomplete|
|          5| Vikram Rao|   Mumbai|  Platinum|Not Provided| Not Provided|           Flight|        High|             Incomplete|
+-----------+-----------+---------+----------+------------+-------------+-----------------+------------+-----------------------+



In [145]:
flat_df.filter((flat_df.preferred_service=="Not Selected") ).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          4|Ayesha Begum|Hyderabad|  Standard|9876500014|ayesha@mail.com|     Not Selected|         Low|             Incomplete|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+

